# Project: LoRA from scratch 🔧
**Low-rank adaptation** is how people fine-tune billion-parameter LLMs on a laptop: freeze the pretrained weight $W$ and learn a small update $\Delta W = BA$ with rank $r \ll d$. Pretrain a network on digits 0–4, then adapt it to digits 5–9 three ways (full fine-tuning, head only, LoRA) and compare accuracy per trainable parameter.

Topic: [[Fine-tuning and Alignment]], [[LLMs Overview]] · guide note: [[Project - LoRA from Scratch]] · paper: [Hu et al. 2021](https://arxiv.org/abs/2106.09685)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import copy, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
X, y = load_digits(return_X_y=True); X = torch.tensor(X / 16, dtype=torch.float32); y = torch.tensor(y)
A_mask, B_mask = y < 5, y >= 5
def split(mask, seed=0):
    idx = torch.where(mask)[0]; g = torch.Generator().manual_seed(seed); idx = idx[torch.randperm(len(idx), generator=g)]; k = int(0.7 * len(idx)); return idx[:k], idx[k:]
A_tr, A_te = split(A_mask); B_tr, B_te = split(B_mask)
base = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
def fit(model, idx, epochs=60, lr=1e-3):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for _ in range(epochs):
        for b_ in idx[torch.randperm(len(idx))].split(64):
            loss = F.cross_entropy(model(X[b_]), y[b_]); opt.zero_grad(); loss.backward(); opt.step()
def acc(model, idx):
    with torch.no_grad(): return (model(X[idx]).argmax(1) == y[idx]).float().mean().item()
fit(base, A_tr); print(f"pretrained on digits 0–4: accuracy {acc(base, A_te):.1%} on 0–4, {acc(base, B_te):.1%} on 5–9 (never seen)")

## 1. The LoRA layer\nWrap a frozen `nn.Linear`: output $= W_0x + \frac{\alpha}{r}BAx$ with $A\in\mathbb R^{r\times d_{in}}$ (small random init) and $B\in\mathbb R^{d_{out}\times r}$ initialised to **zero**, so the adapted model starts *exactly* as the pretrained one.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01); self.B = nn.Parameter(torch.zeros(base.out_features, r)); self.scale = alpha / r
    def forward(self, x):
        # TODO 1: frozen base output + scaled low-rank update (x Aᵀ Bᵀ)
        raise NotImplementedError  # TODO 1

## 2. Count trainable parameters
Return the number of parameters with `requires_grad=True`.

In [ ]:
def n_trainable(model):
    # TODO 2
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_lora():
    lin = nn.Linear(6, 5); lo = LoRALinear(copy.deepcopy(lin), r=2); x = torch.randn(3, 6)
    same = torch.allclose(lo(x), lin(x))
    with torch.no_grad(): lo.B.fill_(1.0)
    changed = torch.allclose(lo(x), lin(x) + (x @ lo.A.T @ lo.B.T) * lo.scale)
    return same and changed and n_trainable(lo) == 2 * 6 + 5 * 2
check("LoRA layer", _t_lora, "With B = 0 the output equals the base layer; trainable params = r·d_in + d_out·r = 22.")
def with_lora(model, r):
    m = copy.deepcopy(model)
    for p in m.parameters(): p.requires_grad = False
    for i in (0, 2, 4): m[i] = LoRALinear(m[i], r=r)
    return m
res = {}
if ready("LoRA layer"):
    full = copy.deepcopy(base); fit(full, B_tr); res["full fine-tune"] = (acc(full, B_te), acc(full, A_te), n_trainable(full))
    head = copy.deepcopy(base)
    for p in head.parameters(): p.requires_grad = False
    for p in head[4].parameters(): p.requires_grad = True
    fit(head, B_tr); res["last layer only"] = (acc(head, B_te), acc(head, A_te), n_trainable(head))
    for r in (1, 4, 16):
        torch.manual_seed(0); m = with_lora(base, r); fit(m, B_tr, lr=3e-3); res[f"LoRA r={r}"] = (acc(m, B_te), acc(m, A_te), n_trainable(m))
    total = sum(p.numel() for p in base.parameters())
    print(f"{'method':<16} {'acc 5–9':>8} {'acc 0–4':>8} {'trainable':>10}")
    for k, (b5, a0, p) in res.items(): print(f"{k:<16} {b5:>8.1%} {a0:>8.1%} {p:>7,} ({p / total:.1%})")
    check("LoRA is efficient", lambda: res["LoRA r=4"][0] > res["full fine-tune"][0] - 0.03 and res["LoRA r=4"][2] < 0.1 * res["full fine-tune"][2],
          "LoRA r=4 should get within 3 points of full fine-tuning with < 10 % of the trainable parameters.")
    print("→ Bonus of LoRA: drop the adapters and you have the original model back (and its 0–4 skills), or keep several adapters for several tasks.")

<details><summary>▶ Solution</summary>

```python
    def forward(self, x):   # LoRALinear
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale
```

```python
def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
```
</details>

## Stretch goals
- **Merge** the adapter: $W \leftarrow W_0 + \frac{\alpha}{r}BA$, then check the merged plain `nn.Linear` gives identical outputs (zero inference overhead).
- Apply LoRA to the attention projections of [[Project - Build a Transformer from Scratch]] and adapt it to a new date format.
- Plot accuracy vs. rank r = 1…64: how low can you go?